# Gold Salary Trends Mart

**Purpose**: Dashboard-ready salary trends with percentile distributions.

**Target Table**: `workspace.gold.gold_salary_trends`

**Grain**: Date + sector/role/location combinations with salary statistics

**Refresh Strategy**: Full refresh (CREATE OR REPLACE) with metadata logging

**Parameters**:
- `lookback_days`: Number of days of historical data (default: 365)
- `force_full_refresh`: Boolean flag to force table recreation
- `min_sample_size`: Minimum observations for reliable statistics (default: 5)

**Key Metrics**:
- Median salaries (min and max)
- Percentile distributions (P25, P75, P90)
- Salary ranges and averages
- Period-over-period changes
- Observation counts

**Data Sources**:
- `workspace.warehouse.fact_salary`

**Metadata Logging**:
- Tracks run history in `workspace.metadata.gold_salary_trends_refresh_log`
- Captures: rows processed, unique dates/sectors/roles/locations, processing time, status

In [0]:
# Configuration
target_table = "workspace.gold.gold_salary_trends"
metadata_table = "workspace.metadata.gold_salary_trends_refresh_log"

# Parameters
lookback_days = 365  # How many days of history to include
force_full_refresh = False  # Set to True to drop and recreate table
min_sample_size = 5  # Minimum observations for reliable statistics

In [0]:
import uuid
from datetime import datetime

# Generate unique run ID
run_id = str(uuid.uuid4())
run_timestamp = datetime.now()
status = "RUNNING"
error_message = None

print(f"Run ID: {run_id}")
print(f"Run Timestamp: {run_timestamp}")
print(f"Lookback Days: {lookback_days}")
print(f"Force Full Refresh: {force_full_refresh}")
print(f"Min Sample Size: {min_sample_size}")

In [0]:
%sql
-- Create table with all columns if it doesn't exist
CREATE TABLE IF NOT EXISTS workspace.metadata.gold_salary_trends_refresh_log (
  run_id STRING NOT NULL,
  run_timestamp TIMESTAMP NOT NULL,
  rows_inserted BIGINT,
  status STRING NOT NULL,
  lookback_days INT,
  rows_processed BIGINT,
  unique_dates INT,
  unique_sectors INT,
  unique_roles INT,
  unique_locations INT,
  force_full_refresh BOOLEAN,
  processing_time_seconds DECIMAL(10,2),
  error_message STRING
)
USING DELTA
COMMENT 'Audit log for gold_salary_trends refresh operations';

In [0]:
# Optionally drop table for full refresh
if force_full_refresh:
    print(f"Force full refresh enabled - dropping table {target_table}")
    spark.sql(f"DROP TABLE IF EXISTS {target_table}")
    print("✓ Table dropped")
else:
    print("Proceeding with CREATE OR REPLACE (preserves table properties)")

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.gold.gold_salary_trends (
  salary_date_sk INT NOT NULL COMMENT 'Date key YYYYMMDD',
  sector_sk BIGINT COMMENT 'FK to dim_sector (NULL for rollups)',
  role_sk BIGINT COMMENT 'FK to dim_role (NULL for rollups)',
  location_sk BIGINT COMMENT 'FK to dim_location (NULL for rollups)',
  salary_min_median DECIMAL(18,2) COMMENT 'Median minimum salary',
  salary_max_median DECIMAL(18,2) COMMENT 'Median maximum salary',
  salary_min_p25 DECIMAL(18,2) COMMENT '25th percentile min salary',
  salary_min_p75 DECIMAL(18,2) COMMENT '75th percentile min salary',
  salary_max_p25 DECIMAL(18,2) COMMENT '25th percentile max salary',
  salary_max_p75 DECIMAL(18,2) COMMENT '75th percentile max salary',
  salary_max_p90 DECIMAL(18,2) COMMENT '90th percentile max salary',
  salary_range_median DECIMAL(18,2) COMMENT 'Median salary range',
  observation_count BIGINT COMMENT 'Number of salary observations'
)
USING DELTA
COMMENT 'Salary trends with percentile distributions'
TBLPROPERTIES (
  'delta.enableChangeDataFeed' = 'true',
  'delta.autoOptimize.optimizeWrite' = 'true',
  'delta.autoOptimize.autoCompact' = 'true'
);

In [0]:
from datetime import datetime, timedelta

# Calculate cutoff date
cutoff_date = int((datetime.now() - timedelta(days=lookback_days)).strftime("%Y%m%d"))

print(f"Building salary trends with aggregations (lookback: {lookback_days} days, cutoff: {cutoff_date})...")

# Insert aggregated salary data
insert_sql = f"""
INSERT OVERWRITE TABLE {target_table}

WITH salary_base AS (
  SELECT
    fs.observation_date_sk AS salary_date_sk,
    j.sector_sk,  -- Get sector from dim_job, not fact_salary
    fs.role_sk,
    fs.location_sk,
    fs.salary_min,
    fs.salary_max
  FROM workspace.warehouse.fact_salary fs
  INNER JOIN workspace.warehouse.dim_job j ON fs.job_sk = j.job_sk AND j.is_current = TRUE
  WHERE fs.observation_date_sk >= {cutoff_date}
    AND fs.salary_period = 'ANNUAL'
    AND fs.salary_currency = 'USD'
    AND fs.salary_min IS NOT NULL
    AND fs.salary_max IS NOT NULL
    AND fs.salary_min > 0
    AND fs.salary_max > 0
),

-- Sector rollup
sector_agg AS (
  SELECT
    salary_date_sk,
    sector_sk,
    CAST(NULL AS BIGINT) AS role_sk,
    CAST(NULL AS BIGINT) AS location_sk,
    PERCENTILE(salary_min, 0.50) AS salary_min_median,
    PERCENTILE(salary_max, 0.50) AS salary_max_median,
    PERCENTILE(salary_min, 0.25) AS salary_min_p25,
    PERCENTILE(salary_min, 0.75) AS salary_min_p75,
    PERCENTILE(salary_max, 0.25) AS salary_max_p25,
    PERCENTILE(salary_max, 0.75) AS salary_max_p75,
    PERCENTILE(salary_max, 0.90) AS salary_max_p90,
    COUNT(*) AS observation_count
  FROM salary_base
  GROUP BY salary_date_sk, sector_sk
  HAVING COUNT(*) >= {min_sample_size}
),

-- Role rollup
role_agg AS (
  SELECT
    salary_date_sk,
    CAST(NULL AS BIGINT),
    role_sk,
    CAST(NULL AS BIGINT),
    PERCENTILE(salary_min, 0.50),
    PERCENTILE(salary_max, 0.50),
    PERCENTILE(salary_min, 0.25),
    PERCENTILE(salary_min, 0.75),
    PERCENTILE(salary_max, 0.25),
    PERCENTILE(salary_max, 0.75),
    PERCENTILE(salary_max, 0.90),
    COUNT(*)
  FROM salary_base
  GROUP BY salary_date_sk, role_sk
  HAVING COUNT(*) >= {min_sample_size}
),

-- Location rollup
location_agg AS (
  SELECT
    salary_date_sk,
    CAST(NULL AS BIGINT),
    CAST(NULL AS BIGINT),
    location_sk,
    PERCENTILE(salary_min, 0.50),
    PERCENTILE(salary_max, 0.50),
    PERCENTILE(salary_min, 0.25),
    PERCENTILE(salary_min, 0.75),
    PERCENTILE(salary_max, 0.25),
    PERCENTILE(salary_max, 0.75),
    PERCENTILE(salary_max, 0.90),
    COUNT(*)
  FROM salary_base
  GROUP BY salary_date_sk, location_sk
  HAVING COUNT(*) >= {min_sample_size}
),

-- Combine all aggregations
combined AS (
  SELECT * FROM sector_agg
  UNION ALL
  SELECT * FROM role_agg
  UNION ALL
  SELECT * FROM location_agg
)

SELECT
  salary_date_sk,
  sector_sk,
  role_sk,
  location_sk,
  CAST(salary_min_median AS DECIMAL(18,2)),
  CAST(salary_max_median AS DECIMAL(18,2)),
  CAST(salary_min_p25 AS DECIMAL(18,2)),
  CAST(salary_min_p75 AS DECIMAL(18,2)),
  CAST(salary_max_p25 AS DECIMAL(18,2)),
  CAST(salary_max_p75 AS DECIMAL(18,2)),
  CAST(salary_max_p90 AS DECIMAL(18,2)),
  CAST((salary_max_median - salary_min_median) AS DECIMAL(18,2)) AS salary_range_median,
  observation_count
FROM combined
"""

spark.sql(insert_sql)
print("✓ Data loaded")

In [0]:
import time

start_time = time.time()

try:
    # Get summary metrics from the gold table
    metrics = spark.sql(f"""
        SELECT 
            COUNT(*) as rows_processed,
            COUNT(DISTINCT salary_date_sk) as unique_dates,
            COUNT(DISTINCT sector_sk) as unique_sectors,
            COUNT(DISTINCT role_sk) as unique_roles,
            COUNT(DISTINCT location_sk) as unique_locations
        FROM {target_table}
    """).collect()[0]
    
    rows_processed = metrics['rows_processed']
    unique_dates = metrics['unique_dates']
    unique_sectors = metrics['unique_sectors']
    unique_roles = metrics['unique_roles']
    unique_locations = metrics['unique_locations']
    processing_time = time.time() - start_time
    status = "SUCCESS"
    
    print(f"✓ Processed {rows_processed:,} rows")
    print(f"✓ Unique dates: {unique_dates}")
    print(f"✓ Unique sectors: {unique_sectors}")
    print(f"✓ Unique roles: {unique_roles}")
    print(f"✓ Unique locations: {unique_locations}")
    print(f"✓ Processing time: {processing_time:.2f}s")
    
except Exception as e:
    status = "FAILED"
    error_message = str(e)
    processing_time = time.time() - start_time
    rows_processed = 0
    unique_dates = 0
    unique_sectors = 0
    unique_roles = 0
    unique_locations = 0
    print(f"✗ Error: {error_message}")
    raise

finally:
    # Write metadata log
    spark.sql(f"""
        INSERT INTO {metadata_table} (
            run_id,
            run_timestamp,
            rows_inserted,
            status,
            lookback_days,
            rows_processed,
            unique_dates,
            unique_sectors,
            unique_roles,
            unique_locations,
            force_full_refresh,
            processing_time_seconds,
            error_message
        )
        VALUES (
            '{run_id}',
            TIMESTAMP'{run_timestamp}',
            {rows_processed},
            '{status}',
            {lookback_days},
            {rows_processed},
            {unique_dates},
            {unique_sectors},
            {unique_roles},
            {unique_locations},
            {force_full_refresh},
            {processing_time:.2f},
            {'NULL' if error_message is None else f"'{error_message}'"}
        )
    """)

In [0]:
%sql
-- Overall summary statistics
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT salary_date_sk) AS unique_dates,
  COUNT(DISTINCT sector_sk) AS unique_sectors,
  COUNT(DISTINCT role_sk) AS unique_roles,
  COUNT(DISTINCT location_sk) AS unique_locations,
  MIN(salary_date_sk) AS earliest_date,
  MAX(salary_date_sk) AS latest_date,
  ROUND(AVG(salary_max_median), 0) AS avg_median_salary,
  SUM(observation_count) AS total_observations
FROM workspace.gold.gold_salary_trends;

In [0]:
%sql
-- Check for data quality issues
SELECT
  'Null salary_date_sk' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_salary_trends
WHERE salary_date_sk IS NULL

UNION ALL

SELECT
  'Negative salaries' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_salary_trends
WHERE salary_min_median < 0 
   OR salary_max_median < 0

UNION ALL

SELECT
  'Invalid salary ranges' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_salary_trends
WHERE salary_max_median < salary_min_median

UNION ALL

SELECT
  'Low observation count' AS check_name,
  COUNT(*) AS issue_count
FROM workspace.gold.gold_salary_trends
WHERE observation_count < 5

ORDER BY issue_count DESC;

In [0]:
%sql
-- Breakdown by aggregation level
SELECT 
  CASE 
    WHEN sector_sk IS NOT NULL AND role_sk IS NULL AND location_sk IS NULL THEN 'Sector'
    WHEN sector_sk IS NULL AND role_sk IS NOT NULL AND location_sk IS NULL THEN 'Role'
    WHEN sector_sk IS NULL AND role_sk IS NULL AND location_sk IS NOT NULL THEN 'Location'
    ELSE 'Other'
  END AS rollup_level,
  COUNT(*) AS row_count,
  COUNT(DISTINCT salary_date_sk) AS unique_dates,
  ROUND(AVG(salary_max_median), 0) AS avg_median_salary,
  SUM(observation_count) AS total_observations
FROM workspace.gold.gold_salary_trends
GROUP BY rollup_level
ORDER BY row_count DESC;

In [0]:
%sql
-- Top 10 sectors by median max salary (most recent date)
SELECT
  ds.sector_name,
  st.salary_date_sk,
  st.salary_max_median,
  st.salary_min_median,
  st.salary_range_median,
  st.observation_count
FROM workspace.gold.gold_salary_trends st
LEFT JOIN workspace.warehouse.dim_sector ds ON st.sector_sk = ds.sector_sk
WHERE st.sector_sk IS NOT NULL
  AND st.role_sk IS NULL
  AND st.location_sk IS NULL
  AND st.salary_date_sk = (SELECT MAX(salary_date_sk) FROM workspace.gold.gold_salary_trends)
ORDER BY st.salary_max_median DESC
LIMIT 10;

In [0]:
%sql
-- View last 10 refresh runs
SELECT 
  run_id,
  run_timestamp,
  lookback_days,
  rows_processed,
  unique_dates,
  unique_sectors,
  unique_roles,
  unique_locations,
  processing_time_seconds,
  status,
  error_message
FROM workspace.metadata.gold_salary_trends_refresh_log
ORDER BY run_timestamp DESC
LIMIT 10;